## Silver Work Increamental 

inreamnetal silver processing using only new bronze rows 

##Step --1 Imports and setup
import sparks, delta ,windows 

In [0]:
from pyspark.sql import functions as f
from pyspark.sql.window import Window
from delta.tables import DeltaTable
from datetime import datetime
import uuid

In [0]:
spark.sql("USE CATALOG `novacart_ecomdatabricks`")
spark.sql("CREATE SCHEMA IF NOT EXISTS silver_schema")

silver_run_id = str(uuid.uuid4())
print("Current Silver Run_id:",silver_run_id)

## Step --2 Silver COntrol Table

it stores the latest silver processing state for each entity .

- the latest bronze run already processed by silver
- latest bronze ingestion timestamp already proceessed
- how many rows were merged in the latest silver run

In [0]:
spark.sql("""
          Create table if not exists novacart_ecomdatabricks.silver_schema.processing_control(

          layer string,
          entity_name string,
          last_processed_bronze_run_id string,
          last_processed_bronze_ingested_at timestamp,
          rows_merged bigint,
          run_status string,
          silver_run_id string,
          updated_at timestamp
          ) USING DELTA
          """)

##Step 3-- Helper Functions 

- upsert to silver: mergescleaned/transformed rows into the silver target table 
- get last ingested or processed data : reads silver watermark
- upserts silver control() : updates the silver controil table 
- get incremantal _bronze() reads only new bronze rows taht silver has not processed yet 

In [0]:
##upsert to silver: mergescleaned/transformed rows into the silver target table

def upsert_to_silver(df_source, target_table, join_key):
    # Check if silver table exists
    # If business key is new, insert; if not, update.
    if spark.catalog.tableExists(target_table):
        dt = DeltaTable.forName(spark, target_table)
        (
            dt.alias("target")
            .merge(
                df_source.alias("source"),
                f"target.{join_key} = source.{join_key}"
            )
            .whenMatchedUpdateAll()
            .whenNotMatchedInsertAll()
            .execute()
        )
    else:
        # Create new silver table if it does not exist
        df_source.write.format("delta").saveAsTable(target_table)


In [0]:
##get last ingested or processed data : reads silver watermark

def get_last_processed_bronze_ingested_at(entity_name: str):
    cntrl= spark.table("novacart_ecomdatabricks.silver_schema.processing_control")\
        .filter(
            (f.col("layer") == "silver")&
            (f.col("entity_name") == entity_name)&
            (f.col("run_status") == "SUCCESS")
        )\
        .orderBy(f.col("updated_at").desc())\
        .limit(1)

    row = cntrl.collect()
    if not row:
        return None
    else:
        return row[0]["last_processed_bronze_ingested_at"]

In [0]:
## upsert_silver_control table :

def upsert_silver_control(entity_name, last_processed_bronze_run_id,last_processed_bronze_ingested_at,rows_merged):
    cntrl_df= spark.createDataFrame(
        [
            (
                "silver",
                entity_name,
                last_processed_bronze_run_id,
                last_processed_bronze_ingested_at,
                int(rows_merged),
                "SUCCESS",
                silver_run_id,
                datetime.utcnow()
            )
        ],
        schema = """
        layer string,
        entity_name string,
        last_processed_bronze_run_id string,
        last_processed_bronze_ingested_at timestamp,
        rows_merged bigint,
        run_status string,
        silver_run_id string,
        updated_at timestamp
       """
    )

    dt= DeltaTable.forName(spark,"novacart_ecomdatabricks.silver_schema.processing_control")
    (dt.alias("t")
     .merge(cntrl_df.alias("s"),f"t.entity_name = s.entity_name and t.layer = s.layer")
     .whenMatchedUpdate(set={
         "last_processed_bronze_run_id": "s.last_processed_bronze_run_id",
         "last_processed_bronze_ingested_at": "s.last_processed_bronze_ingested_at",
         "rows_merged": "s.rows_merged",
         "run_status": "s.run_status",
         "silver_run_id": "s.silver_run_id",
         "updated_at": "s.updated_at"
      }
     )
     .whenNotMatchedInsertAll()
     .execute()
    )

In [0]:
## it will read only the bronze ingestion which has not read by silver:

def  get_incremental_bronze(bronze_table,entity_name):
    last_ingested_at = get_last_processed_bronze_ingested_at(entity_name)

    bronze_df= spark.read.table(bronze_table)
    if last_ingested_at is None:
        return bronze_df,last_ingested_at
    else:
        return bronze_df.filter(f.col("bronze-ingested_at")>f.lit(last_ingested_at)), last_ingested_at

Step: 4 -- Orders Incremantal Processing

this cells process orders from bronze to silver:

 - reads only new bronze order rows
 - cleans values like order_sattus and order_amount
 - keep only latest version per order_id
 - validate  buisness rules
 - sends bad rows to quarantine
 - merge good rows into order_transformed 

In [0]:
df_orders= spark.sql("select * from novacart_ecomdatabricks.bronze_schema.orders_raw")

df_orders.show()

In [0]:
df_payment= spark.sql("select * from novacart_ecomdatabricks.bronze_schema.payments_raw")

df_payment.show()

In [0]:
df_product= spark.sql("select * from novacart_ecomdatabricks.bronze_schema.products_raw")

df_product.show()